# Part 4: Sequential Meal Modeling with Item2Vec

In the food delivery context, the *sequence* of items added to a cart carries immense semantic meaning. For example, `[Biryani -> Salan -> Coke]` represents a completed meal, whereas `[Biryani]` is incomplete. 

To capture this sequential logic, we adapted Word2Vec into **Item2Vec**. By treating historical *orders* as *sentences* and *items* as *words*, we map every menu item into a dense 32-dimensional embedding space where complementary items cluster together.

In [ ]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

## 1. Constructing the 'Sentences'
We load the temporal order logs. For each unique `order_id`, we sort the items by the time they were added to the cart to form a sequence.

In [ ]:
try:
    orders_df = pd.read_parquet("../data/raw/orders_exploded_adv.parquet")
    print(f"Loaded {len(orders_df)} raw order events.")
    
    # Sort chronologically to preserve the cart addition sequence
    orders_df = orders_df.sort_values(['order_id', 'order_time'])
    
    # Group by order to create sentences
    sentences = orders_df.groupby('order_id')['item_id'].apply(lambda x: [str(item) for item in x]).tolist()
    print(f"Constructed {len(sentences)} training sequences.")
except Exception as e:
    print(f"Error loading data: {e}")
    # Mock data for demonstration
    sentences = [['101', '105', '201'], ['102', '305'], ['101', '201']]

## 2. Training the Item2Vec Model
We use Skip-Gram architecture. Since cart sizes are generally small (2-5 items), we use a small `window` size to ensure strong local co-occurrence learning.

In [ ]:
print("Training Item2Vec model...")
item2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=32,    # 32-D dense embeddings
    window=3,          # Context window (small for carts)
    min_count=2,       # Minimum frequency 
    workers=4,         # Parallel training
    sg=1,              # Skip-gram (better for infrequent items than CBOW)
    epochs=20          # Extended epochs for stability
)
print("Training complete.")

## 3. Extracting Similarity Scores
When a user requests a recommendation, the LightGBM ranker calculates the `cosine_similarity` between the candidate item and the *average embedding of the current cart*. Let's extract the raw vector space to visualize its quality.

In [ ]:
# Extract the learned embeddings into a dictionary
item_embeddings = {item_id: item2vec_model.wv[item_id] for item_id in item2vec_model.wv.index_to_key}
print(f"Extracted {len(item_embeddings)} item vectors.")

# Example: Finding most similar items to Item '11'
if '11' in item2vec_model.wv:
    similar_items = item2vec_model.wv.most_similar('11', topn=3)
    print("Items most frequently bought with Item 11:", similar_items)

## 4. Visualizing the Embedding Space (PCA)
We reduce the 32-D space to 2-D to visualize if Mains, Desserts, and Beverages cluster sensibly.

In [ ]:
# Get items and vectors
items = list(item_embeddings.keys())
vectors = np.array([item_embeddings[item] for item in items])

if len(vectors) > 1:
    pca = PCA(n_components=2)
    vectors_2d = pca.fit_transform(vectors)
    
    plt.figure(figsize=(10, 8))
    # Plot a sample of items to avoid overcrowding
    sample_size = min(200, len(items))
    plt.scatter(vectors_2d[:sample_size, 0], vectors_2d[:sample_size, 1], alpha=0.6, color='teal')
    
    plt.title('2D PCA Projection of Item2Vec Embeddings')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.show()

## Conclusion
These embeddings are pickled to `data/processed/item_embeddings.pkl`. At runtime, the `cart_stage.py` module computes the live semantic distance between the active cart and the 8 candidate add-ons retrieved by Redis, offering LightGBM a powerful sequential signal.